####Imports

In [0]:
from pyspark.sql.functions import *
from datetime import datetime
import time

####Table configuration

In [0]:
bronze_table = (
    "workspace.nyc_taxi_aws.bronze_taxi_trips"
)

silver_table = (
    "workspace.nyc_taxi_aws.silver_taxi_trips_curated"
)

data_quality_table = (
    "workspace.nyc_taxi_aws.data_quality_aws"
)

gold_daily_table = (
    "workspace.nyc_taxi_aws.gold_daily_metrics_aws"
)

gold_hourly_table = (
    "workspace.nyc_taxi_aws.gold_hourly_metrics_aws"
)

gold_payment_table = (
    "workspace.nyc_taxi_aws.gold_payment_metrics_aws"
)

gold_vendor_table = (
    "workspace.nyc_taxi_aws.gold_vendor_metrics_aws"
)

monitoring_table = (
    "workspace.nyc_taxi_aws.pipeline_monitoring"
)

gold_dashboard_metrics_aws=(
    "workspace.nyc_taxi_aws.gold_dashboard_metrics_aws"
)


####Generate pipeline run ID

In [0]:
run_id = datetime.now().strftime(
    "%Y%m%d_%H%M%S"
)

pipeline_start_time = time.time()

print(
    f"Pipeline monitoring started"
)

print(
    f"Run ID: {run_id}"
)

Pipeline monitoring started
Run ID: 20260729_204316


####Count Bronze records

In [0]:
bronze_count = (
    spark.table(
        bronze_table
    ).count()
)

print(
    f"Bronze records: {bronze_count}"
)

Bronze records: 36165446


####Count Silver records

In [0]:
silver_count = (
    spark.table(
        silver_table
    ).count()
)

print(
    f"Silver records: {silver_count}"
)

Silver records: 33564513


####Check Data Quality

In [0]:
failed_quality_checks = (
    spark.table(
        data_quality_table
    )
    .filter(
        col("status") == "FAIL"
    )
    .count()
)

total_quality_checks = (
    spark.table(
        data_quality_table
    ).count()
)

if failed_quality_checks == 0:

    data_quality_status = "SUCCESS"

else:

    data_quality_status = "FAILED"

print(
    f"Data Quality status: "
    f"{data_quality_status}"
)

print(
    f"Failed checks: "
    f"{failed_quality_checks}"
)

Data Quality status: SUCCESS
Failed checks: 0


####Count Gold records

In [0]:
gold_daily_count = (
    spark.table(
        gold_daily_table
    ).count()
)

gold_hourly_count = (
    spark.table(
        gold_hourly_table
    ).count()
)

gold_payment_count = (
    spark.table(
        gold_payment_table
    ).count()
)

gold_vendor_count = (
    spark.table(
        gold_vendor_table
    ).count()
)

gold_dashboard_metrics_aws_count = (
    spark.table(
        gold_dashboard_metrics_aws
    ).count()
)

print(
    f"Gold daily records: "
    f"{gold_daily_count}"
)

print(
    f"Gold hourly records: "
    f"{gold_hourly_count}"
)

print(
    f"Gold payment records: "
    f"{gold_payment_count}"
)

print(
    f"Gold vendor records: "
    f"{gold_vendor_count}"
)

print(
    f"GOLD_DASHBOARD: "
    f"{gold_dashboard_metrics_aws_count}"
)

Gold daily records: 278
Gold hourly records: 24
Gold payment records: 5
Gold vendor records: 3
GOLD_DASHBOARD: 278


####Calculate total processing time

In [0]:
pipeline_end_time = time.time()

duration_seconds = (
    pipeline_end_time
    - pipeline_start_time
)

print(
    f"Monitoring duration: "
    f"{duration_seconds:.2f} seconds"
)

Monitoring duration: 135.90 seconds


####Create monitoring records

In [0]:
monitoring_data = [
    (
        run_id,
        "BRONZE",
        "SUCCESS",
        bronze_count,
        datetime.now(),
        duration_seconds,
        None
    ),

    (
        run_id,
        "SILVER",
        "SUCCESS",
        silver_count,
        datetime.now(),
        duration_seconds,
        None
    ),

    (
        run_id,
        "DATA_QUALITY",
        data_quality_status,
        total_quality_checks,
        datetime.now(),
        duration_seconds,
        (
            f"{failed_quality_checks} "
            f"failed checks"
            if failed_quality_checks > 0
            else None
        )
    ),

    (
        run_id,
        "GOLD_DAILY",
        "SUCCESS",
        gold_daily_count,
        datetime.now(),
        duration_seconds,
        None
    ),

    (
        run_id,
        "GOLD_HOURLY",
        "SUCCESS",
        gold_hourly_count,
        datetime.now(),
        duration_seconds,
        None
    ),

    (
        run_id,
        "GOLD_PAYMENT",
        "SUCCESS",
        gold_payment_count,
        datetime.now(),
        duration_seconds,
        None
    ),

    (
        run_id,
        "GOLD_VENDOR",
        "SUCCESS",
        gold_vendor_count,
        datetime.now(),
        duration_seconds,
        None
    ),
    (
        run_id,
        "GOLD_DASHBOARD",
        "SUCCESS",
        gold_dashboard_metrics_aws_count,
        datetime.now(),
        duration_seconds,
        None
    )
]

####Create DataFrame

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, LongType, TimestampType, DoubleType

monitoring_schema = StructType([
    StructField("run_id", StringType(), True),
    StructField("layer_name", StringType(), True),
    StructField("status", StringType(), True),
    StructField("records_processed", LongType(), True),
    StructField("execution_timestamp", TimestampType(), True),
    StructField("duration_seconds", DoubleType(), True),
    StructField("error_message", StringType(), True)
])

monitoring_df = spark.createDataFrame(
    monitoring_data,
    monitoring_schema
)

display(
    monitoring_df
)

run_id,layer_name,status,records_processed,execution_timestamp,duration_seconds,error_message
20260729_204316,BRONZE,SUCCESS,36165446,2026-07-29T20:45:35.314Z,135.8953878879547,null
20260729_204316,SILVER,SUCCESS,33564513,2026-07-29T20:45:35.314Z,135.8953878879547,null
20260729_204316,DATA_QUALITY,SUCCESS,35,2026-07-29T20:45:35.314Z,135.8953878879547,null
20260729_204316,GOLD_DAILY,SUCCESS,278,2026-07-29T20:45:35.314Z,135.8953878879547,null
20260729_204316,GOLD_HOURLY,SUCCESS,24,2026-07-29T20:45:35.314Z,135.8953878879547,null
20260729_204316,GOLD_PAYMENT,SUCCESS,5,2026-07-29T20:45:35.314Z,135.8953878879547,null
20260729_204316,GOLD_VENDOR,SUCCESS,3,2026-07-29T20:45:35.314Z,135.8953878879547,null
20260729_204316,GOLD_DASHBOARD,SUCCESS,278,2026-07-29T20:45:35.314Z,135.8953878879547,null


####Write monitoring data

In [0]:
(
    monitoring_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(
        monitoring_table
    )
)